In [1]:
import os
import random
import numpy as np
import scipy.io as spio

def phi(inVolt,threshold):
    currArr = inVolt/threshold
    return np.where(currArr>1,1,currArr)

#Implementation of the Galves–Löcherbach model for biophysically simulating real spike trains
#doi:10.1007/s10955-013-0733-9
def galvesLocherbachSimulator(unitCount,connGraph,remain,thresh,timeSteps):
    #Unit Count, Connection Graph, and timeSteps are all known
    random.seed(2022) #Pick a RNG seed, does not matter which one, just make sure it consistent across all runs
    outputSimmedSpikeTrain = 0
    sampV = np.random.randint(0,thresh,size=(unitCount,))
    for t in range(timeSteps):
        spikeProb = phi(sampV,thresh)
        randArr = np.random.uniform(0,thresh*3,size=(unitCount,))
        trueSpike = np.reshape(np.where(spikeProb>randArr,1,0),(len(spikeProb),1)).astype(int)
        if np.shape(outputSimmedSpikeTrain) == ():
            outputSimmedSpikeTrain = trueSpike
        else:
            outputSimmedSpikeTrain = np.concatenate((outputSimmedSpikeTrain,trueSpike),axis=1)
        voltSpike = np.where(spikeProb>randArr,0,spikeProb)
        voltSpike = voltSpike*remain
        sampV = voltSpike+np.sum(voltSpike*connGraph)
    return outputSimmedSpikeTrain.astype(int)

def generateFakeConnections(unitCount):
    randomArr = np.random.random((unitCount,unitCount))
    lowerTriangleInds = np.tril_indices(unitCount)
    randomArr[lowerTriangleInds] = 0
    return randomArr + np.transpose(randomArr)

def convertSimmedSpikesIntoVectorBlock(inputSpikeArr,lastInd):
    indArr = np.arange(0,np.shape(inputSpikeArr)[1],6.6).astype(int)
    gammaBlock = 0
    for i in range(len(indArr)):
        if i == len(indArr) - 1:
            lowInd = indArr[i]
            currBlock = inputSpikeArr[:,lowInd:]
            currArr = np.sum(currBlock,axis=1)/.033
            currArrRes = np.reshape(currArr,(len(currArr),1))
            gammaBlock = np.concatenate((gammaBlock,currArrRes),axis=1)
        else:
            lowInd = indArr[i]
            higInd = indArr[int(i+1)]
            currBlock = inputSpikeArr[:,lowInd:higInd]
            currArr = np.sum(currBlock,axis=1)/.033
            currArrRes = np.reshape(currArr,(len(currArr),1))
            if i == 0:
                gammaBlock = currArrRes
            else:
                gammaBlock = np.concatenate((gammaBlock,currArrRes),axis=1)
    gammaBlockUpd = gammaBlock[:,-lastInd:]
    return gammaBlockUpd

def convertGammaToPosition(inputVectorBlock,inputFields):
    positionsX = []
    positionsY = []
    for col in range(np.shape(inputVectorBlock)[1]):
        currVector = inputVectorBlock[:,col]
        singleTimestampArr = np.zeros((128,128))
        for un,val in enumerate(currVector):
            currFieldArr = val*inputFields[un]
            singleTimestampArr += currFieldArr
        maxX,maxY = np.unravel_index(singleTimestampArr.argmax(),singleTimestampArr.shape)
        positionsX.append(maxX)
        positionsY.append(maxY)
    return positionsX,positionsY

def loadValuesFromRefFile(refFileOfInterest):
    fakeX = [None]
    fakeY = [None]
    refmat = spio.loadmat(refFileOfInterest)
    unitFields = refmat.get('unitFields')
    spikeThresh,voltRemain = refmat.get('modelValuesAll').flatten()
    for i in range(10):
        connectionAll = generateFakeConnections(len(refmat.get('regionIndex').flatten()))
        simmedSpikes = galvesLocherbachSimulator(np.shape(connectionAll)[0],connectionAll,voltRemain,spikeThresh,1923)
        gammaBlock = convertSimmedSpikesIntoVectorBlock(simmedSpikes,111)
        estimatedX,estimatedY = convertGammaToPosition(gammaBlock,unitFields)
        fakeX.append(estimatedX)
        fakeY.append(estimatedY)
    return {'fakePosX':fakeX[1:],'fakePosY':fakeY[1:]}

refFile = ''
savFile = ''
dictToSave = loadValuesFromRefFile(refFile)
var = spio.savemat(savFile,dictToSave)
print('Done')


Done
